In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import canny
from skimage.draw import circle_perimeter
from scipy.ndimage import gaussian_filter
from scipy.optimize import leastsq
import time

%matplotlib inline

In [ ]:
# Define the base paths
training_base_path = 'database/training/'
testing_base_path = 'database/testing/'

In [ ]:
def evaluate_seed_point(seed_finder):
    start_time = time.time()

    # Evaluate the seed point finder on the training dataset (only mid slice)
    results = []
    patient_folders = [folder for folder in os.listdir(testing_base_path) if os.path.isdir(os.path.join(testing_base_path, folder))]
    print(f"Found {len(patient_folders)} patients in the training and testing dataset.")

    i = 0
    for patient_folder in patient_folders:
    # Try to get the ground truth segmentation (patient001_frame01_gt.nii.gz)
        i += 1
        if i%10 == 0:
            print(f"Processing patient {i}/{len(patient_folders)}")

        # Load the ground truth segmentation
        try:
            gt_path = os.path.join(testing_base_path, patient_folder, f"{patient_folder}_frame01_gt.nii.gz")
            gt = nib.load(gt_path).get_fdata()
            # Get the mid slice
            gt = gt[:, :, gt.shape[-1]//2]
        except:
            print(f"Error loading GT for patient {patient_folder}")
            continue

        # Load the 4d of the patient
        image_4d = os.path.join(testing_base_path, patient_folder, f"{patient_folder}_4d.nii.gz")
        img = nib.load(image_4d).get_fdata()

        # Find the seed point
        seed_point = seed_finder(img)

        # Evaluate the seed point
        seed_x, seed_y = seed_point
        if gt[seed_x, seed_y] == 3:
            results.append(1)
        else:
            results.append(0)

    print(f"Correct: {np.sum(results)} / {len(results)}, Time: {time.time() - start_time}")
    return np.mean(results)

In [ ]:
def estimate_initial_seed(image_4d):
    # Subtract phase 1 and phase max images
    image_orig = image_4d[:, :, :, 0]

    phase_diastole = image_orig[:, :, 0]

    # Find the systole phase (max difference with diastole)
    max_diff = 0
    max_diff_index = 0
    for i in range(1, image_orig.shape[-1]):
        phase = image_orig[:, :, i]
        diff = np.sum(np.abs(phase_diastole - phase))
        if diff > max_diff:
            max_diff = diff
            max_diff_index = i
    
    phase_systole = image_orig[:, :, max_diff_index]
    sub_image = phase_systole - phase_diastole
    
    # Normalize the image using the 99th percentile
    sub_image_norm = sub_image / np.percentile(sub_image, 99)

    # Edge detection
    edges = canny(sub_image_norm, sigma=2, low_threshold=0.01, high_threshold=0.1)
    
    # Perform Hough Transform to detect circles
    hough_radii = np.arange(20, 50, 2)
    hough_res = hough_circle(edges, hough_radii)
    
    # Select the most prominent circles
    accums, cx, cy, radii = hough_circle_peaks(hough_res, hough_radii, total_num_peaks=10)
    
    # Initialize an accumulator for the Gaussian likelihoods
    likelihood_accumulator = np.zeros_like(sub_image, dtype=float)
    
    # Accumulate Gaussian likelihoods for each detected circle
    for center_y, center_x, radius in zip(cy, cx, radii):
        gaussian_likelihood = np.zeros_like(sub_image, dtype=float)
        gaussian_likelihood[center_y, center_x] = 1
        gaussian_likelihood = gaussian_filter(gaussian_likelihood, sigma=radius)
        likelihood_accumulator += gaussian_likelihood
    
    # Determine the seed point based on the maximum likelihood
    seed_point = np.unravel_index(np.argmax(likelihood_accumulator), likelihood_accumulator.shape)
    
    return seed_point

In [ ]:
# Evaluate the seed point finder
print("Evaluating the seed point finder...")
evaluate_seed_point(estimate_initial_seed)